# Solar Filament Segmentation 2026 — end-to-end pipeline

This notebook reproduces the submitted predictions from the stored model weights. See `README.md` for the method description. Training is optional and shown at the end.

In [ ]:
import sys, os
sys.path.insert(0, 'src')
import numpy as np, pandas as pd, cv2, torch, matplotlib.pyplot as plt
from common import RAW, ROOT, load_image, masks_to_instances, panoptic_quality
from predict import load_models, predict_prob
WEIGHTS = ['runs/resnet34_f0/best.pt']
THR, MIN_AREA, TTA = 0.5, 150, True
print(torch.cuda.is_available())

## 1. Data
Training annotations are COCO polygons; `src/prepare.py` rasterizes them into instance maps and soft consensus masks (fraction of annotators marking a pixel).

In [ ]:
if not os.path.exists('data/masks/soft'):
    !python src/prepare.py
splits = pd.read_csv('data/splits.csv'); splits.head()

In [ ]:
stem = splits.stem.iloc[0]
img = load_image(f'{RAW}/train/train_images/{stem}.jpeg')
soft = cv2.imread(f'data/masks/soft/{stem}.png', 0)
fig, ax = plt.subplots(1, 2, figsize=(14, 7)); ax[0].imshow(img, cmap='gray'); ax[0].set_title(stem); ax[1].imshow(soft, cmap='magma'); ax[1].set_title('consensus mask'); plt.show()

## 2. Model and inference
U-Net (ResNet34 encoder, pretrained on ImageNet, adapted to 1 input channel). Full 2048x2048 inference with flip test-time augmentation.

In [ ]:
models = load_models(WEIGHTS, 'cuda')
prob = predict_prob(models, img, 'cuda', TTA)
inst = masks_to_instances(prob, THR, MIN_AREA)
fig, ax = plt.subplots(1, 2, figsize=(14, 7)); ax[0].imshow(prob, cmap='magma'); ax[0].set_title('probability'); ax[1].imshow(inst, cmap='nipy_spectral'); ax[1].set_title(f'{inst.max()} instances'); plt.show()

## 3. Validation (fold 0, grouped by observation date)
Panoptic Quality against every annotator set of each validation image.

In [ ]:
import json
images_df = pd.DataFrame(json.load(open(f'{RAW}/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'))['images'])
va = splits[splits.fold == 0].stem.tolist()
pqs = []
for s in va:
    p = predict_prob(models, load_image(f'{RAW}/train/train_images/{s}.jpeg'), 'cuda', TTA)
    ins = masks_to_instances(p, THR, MIN_AREA)
    for iid in images_df[images_df.file_name == s + '.jpeg'].id:
        gt = cv2.imread(f'data/masks/inst/{iid}.png', cv2.IMREAD_UNCHANGED).astype(np.int32)
        pqs.append(panoptic_quality(gt, ins)[0])
print('validation PQ', np.mean(pqs))

## 4. Test predictions and submission file

In [ ]:
!python src/predict.py --weights {' '.join(WEIGHTS)} --thr {THR} --min_area {MIN_AREA} {'--tta' if TTA else ''} --out submission.csv
!python src/check_submission.py submission.csv
pd.read_csv('submission.csv').head()

## 5. (Optional) Re-train
```
python src/train.py --encoder resnet34 --fold 0 --epochs 40
python src/tune.py --weights runs/resnet34_f0/best.pt --tta
```